In [121]:
from pyspark.sql import SparkSession
from pyspark.sql.functions  import col, expr, when, trim, lower, count, sum, avg, rank, lag, lead, row_number,max
from pyspark.sql.types  import IntegerType, StringType, StructType, StructField,DoubleType
from pyspark.sql.window import Window

In [108]:
spark = SparkSession.builder.appName("Real_world_analogy").master("local[*]").getOrCreate()
sc = spark.sparkContext

print(sc)

<SparkContext master=local[*] appName=Real_world_analogy>


**Section 1 — DataFrame DSL, Select & Filter**

Q1. Load and inspect the data

Q2. Select, rename, and filter

Q3. Complex filter logic

In [109]:
df_customer = spark.read.csv(
    "/content/customers.csv",
    header=True,
    inferSchema=True
)
df_customer.show()

df_customer.printSchema()

+-----------+-------------+----------+------+-------------+---------------+
|customer_id|customer_name|      city|region|customer_tier|account_manager|
+-----------+-------------+----------+------+-------------+---------------+
|       1001|        Aarav|   Chennai| South|      Premium|         Ananya|
|       1002|         Diya|    Mumbai|  West|     Standard|          Rahul|
|       1003|        Kabir| Bengaluru| South|      Premium|         Ananya|
|       1004|        Meera|     Delhi| North|     Standard|         Vikram|
|       1005|       Ishaan| Hyderabad| South|         Gold|          Rahul|
|       1006|        Anika|      Pune|  West|     Standard|         Vikram|
|       1007|        Rohan|   Chennai| South|         Gold|         Ananya|
|       1008|         Sara|   Kolkata|  East|      Premium|         Vikram|
|       1009|        Arjun|    Mumbai|  West|     Standard|          Rahul|
|       1010|         Nila|Coimbatore| South|         Gold|         Ananya|
|       1011

In [110]:
df_customer = spark.read.csv(
    "/content/customers.csv",
    header=True,
    schema=StructType([
        StructField("customer_id", IntegerType(), True),
        StructField("customer_name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("region", StringType(), True),
        StructField("customer_tier", StringType(), True),
        StructField("account_manager",StringType(), True)
    ]))

df_customer.show(10)
df_customer.printSchema()
count = df_customer.count()
print(f"This customer_data has the total count of {count}")

+-----------+-------------+----------+------+-------------+---------------+
|customer_id|customer_name|      city|region|customer_tier|account_manager|
+-----------+-------------+----------+------+-------------+---------------+
|       1001|        Aarav|   Chennai| South|      Premium|         Ananya|
|       1002|         Diya|    Mumbai|  West|     Standard|          Rahul|
|       1003|        Kabir| Bengaluru| South|      Premium|         Ananya|
|       1004|        Meera|     Delhi| North|     Standard|         Vikram|
|       1005|       Ishaan| Hyderabad| South|         Gold|          Rahul|
|       1006|        Anika|      Pune|  West|     Standard|         Vikram|
|       1007|        Rohan|   Chennai| South|         Gold|         Ananya|
|       1008|         Sara|   Kolkata|  East|      Premium|         Vikram|
|       1009|        Arjun|    Mumbai|  West|     Standard|          Rahul|
|       1010|         Nila|Coimbatore| South|         Gold|         Ananya|
+-----------

**Reason for using Explicit Schema**





1.   Why we prefer explicit schema, because the data we are
getting from the raw data could be unclean data so if we use inferschema they will assign on own by reading the data.
2.  However, In the raw data, the data column has some special character then by using the inferschema they will assign the column as a string.

This is the main reason we are using the explicit schmea on our own.


In [111]:
df_sales_orders = spark.read.csv(
    "/content/sales_orders.csv",
    header = True,
    inferSchema= True

)
df_sales_orders.show()
df_sales_orders.printSchema()
sales_count = df_sales_orders.count()
print(f"This sales_order_data has the total count of {sales_count}")

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|
|    5002|       1002|2026-01-04|  Headphones|Electronics|       2|      2500|           0|           UPI|Completed|
|    5003|       1003|2026-01-05|Office Chair|  Furniture|       1|      8500|          10|          Card|Completed|
|    5004|       1001|2026-01-08|    Keyboard|Electronics|       2|      1800|           5|           UPI|Completed|
|    5005|       1004|2026-01-10|        Desk|  Furniture|       1|     12000|          15| Bank Transfer| Returned|
|    5006|       1005|2026-01-12|     Monitor|Electronics|      

In [112]:
df_sales_orders_1 = df_sales_orders.select(
    "order_id",
    "customer_id",
    "product",
    "quantity").filter(
        col("status") == "Completed"
    ).filter(
        col("quantity") >= 2
    ).withColumnRenamed("product", "product_name").orderBy(
        expr("order_id desc")
    )

df_sales_orders_1.show()
df_sales_orders_1.count()

+--------+-----------+------------+--------+
|order_id|customer_id|product_name|quantity|
+--------+-----------+------------+--------+
|    5002|       1002|  Headphones|       2|
|    5004|       1001|    Keyboard|       2|
|    5006|       1005|     Monitor|       2|
|    5007|       1006|Notebook Set|       5|
|    5011|       1010|Notebook Set|      10|
|    5016|       1004|Notebook Set|       4|
|    5019|       1007|     Monitor|       2|
|    5021|       1009|Notebook Set|       8|
|    5022|       1010|    Keyboard|       2|
|    5027|       1003|Notebook Set|       6|
|    5028|       1005|  Headphones|       2|
+--------+-----------+------------+--------+



11

In [113]:
df_filter = df_sales_orders.filter(
    col("order_date").between("2026-02-01", "2026-02-28")
).filter(
    col("category") == "Electronics").filter(
        (col("discount_pct") >= 5 ) |
        (col("status") == "Pending")
        )

df_filter.show()


+--------+-----------+----------+----------+-----------+--------+----------+------------+--------------+---------+
|order_id|customer_id|order_date|   product|   category|quantity|unit_price|discount_pct|payment_method|   status|
+--------+-----------+----------+----------+-----------+--------+----------+------------+--------------+---------+
|    5017|       1005|2026-02-07|    Laptop|Electronics|       1|     65000|           5|          Card|Completed|
|    5018|       1006|2026-02-09|Headphones|Electronics|       3|      2500|          10|           UPI| Returned|
|    5024|       1012|2026-02-25|   Monitor|Electronics|       1|     14500|           5|           UPI|Completed|
+--------+-----------+----------+----------+-----------+--------+----------+------------+--------------+---------+



**Section 2 — Expressions, withColumn & Column Operations**

Q4. Derived business metrics

Q5. Conditional classification

Q6. String and null handling


In [114]:
df_gross = df_sales_orders.withColumn(
    "gross_amount",
    expr("quantity * unit_price")
).withColumn(
    "discount_amount",
    expr("gross_amount * (discount_pct/100)")
).withColumn(
    "net_amount",
    expr("gross_amount - discount_amount")
)

df_gross.show()


+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|
|    5002|       1002|2026-01-04|  Headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|
|    5003|       1003|2026-01-05|Office Chair|  Furniture|       1|      8500|          10|          Card|Completed|        8500|          850.0|    7650.0|
|    5004|       1001|2026-01-08|    Keyboard|Electronics|

In [115]:
df_order_value = df_gross.withColumn(
    "order_value_brand",
    expr("""
    CASE
      WHEN net_amount <= 20000 THEN "High"
      WHEN net_amount <= 5000 THEN "Medium"
      ELSE "Low"
    END
    """)
)

df_order_value.show()



+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|
|    5002|       1002|2026-01-04|  Headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|             High|
|    5003|       1003|2026-01-05|Office Chair|  Furniture|       1|      8500|          10|          Card|Completed|        8

In [116]:
df_order_value_1 = df_gross.withColumn(
    "order_value_brand",
    when(col("net_amount") <= 20000 ,"High")
    .when(col("net_amount") <= 5000 ,"Medium")
    .otherwise("Low")
)

df_order_value.show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|
|    5002|       1002|2026-01-04|  Headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|             High|
|    5003|       1003|2026-01-05|Office Chair|  Furniture|       1|      8500|          10|          Card|Completed|        8

**Safe approach to handle the Null**

In the raw_data_set we can use WHEN condition to check the null value and modify accordingly


In [117]:
df_trim = df_order_value_1.withColumn(
    "product",
    lower(trim(col("product")))
)

df_trim.show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|
|    5002|       1002|2026-01-04|  headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|             High|
|    5003|       1003|2026-01-05|office chair|  Furniture|       1|      8500|          10|          Card|Completed|        8

**Section 3 — Read, Write & File Formats**

Q7. Read options and schema validation

Q8. Write partitioned output

**Handle Malformed**


*   Before validate the Data-set we to verify the file excistience,column header, check schema and data types

*   Values that cannot be parsed into the expected data types.

Best practice: Use an explicit schema and quarantine invalid records instead of silently dropping them.




In [118]:
df_trim.write.mode("overwrite").partitionBy("category").parquet("df_orders_parquet")

Why parquent file is better then the csv, because it consume the less memory and its easy to store and improving query performance when processing large datasets

**Section 4 — Aggregations**

Q9. Multi-level aggregation

Q10. Conditional aggregation

Q11. Customer performance

In [122]:
df_agg = df_trim.groupBy("category").agg(
    count("order_id").alias("order_count"),
    sum("quantity").alias("total_quantity"),
    sum("gross_amount").alias("total_revenue"),
    sum("net_amount").alias("net_revenue"),
    avg("net_amount").alias("avg_net_order_value")
).orderBy(
    col("net_revenue").desc()
)

df_agg.show()

+-----------+-----------+--------------+-------------+-----------+-------------------+
|   category|order_count|total_quantity|total_revenue|net_revenue|avg_net_order_value|
+-----------+-----------+--------------+-------------+-----------+-------------------+
|Electronics|         16|            24|       377800|   359090.0|          22443.125|
|  Furniture|          9|            10|       102500|    95150.0| 10572.222222222223|
| Stationery|          5|            33|         8250|     7900.0|             1580.0|
+-----------+-----------+--------------+-------------+-----------+-------------------+



In [123]:
df_agg_2 = df_trim.groupBy("status").agg(
    count("order_id").alias("order_count"),
    sum("net_amount").alias("net_revenue")
)

df_agg_2.show()


+---------+-----------+-----------+
|   status|order_count|net_revenue|
+---------+-----------+-----------+
| Returned|          3|    18570.0|
|Completed|         24|   405220.0|
|  Pending|          3|    38350.0|
+---------+-----------+-----------+



Explain whether returned orders should be
included in business revenue.

No...we sould not inclued the return order to be inculed why because the return package might have so many reason by the customer so we should sort that and again we have to sell that package

In [124]:
df_joins = df_trim.join(
    df_customer,
    on = "customer_id",
    how = "left"
).filter(
    col("status") == "Completed"
).groupby(
    "customer_id",
    "customer_name",
    "city"

).agg(
    count("order_id").alias("completed_order"),
    sum("net_amount").alias("net_spend")
    ).orderBy(
        col("net_spend").desc()
        )


df_joins.show(5)

+-----------+-------------+---------+---------------+---------+
|customer_id|customer_name|     city|completed_order|net_spend|
+-----------+-------------+---------+---------------+---------+
|       1001|        Aarav|  Chennai|              3| 130170.0|
|       1005|       Ishaan|Hyderabad|              3|  92600.0|
|       1011|       Vivaan|    Delhi|              1|  58500.0|
|       1007|        Rohan|  Chennai|              1|  29000.0|
|       1003|        Kabir|Bengaluru|              3|  19950.0|
+-----------+-------------+---------+---------------+---------+
only showing top 5 rows


**Section 5 — Window Functions**

Q12. Rank within a category

Q13. Running total

Q14. Previous order comparison

Q15. Latest order per customer


In [125]:
df_trim.show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|
|    5002|       1002|2026-01-04|  headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|             High|
|    5003|       1003|2026-01-05|office chair|  Furniture|       1|      8500|          10|          Card|Completed|        8

In [126]:
df_rank = df_trim.groupBy("category", "product").agg(
    sum("net_amount").alias("total_revenue")
).withColumn(
    "rank",
    rank().over(
        Window.partitionBy("category").orderBy(col("total_revenue").desc())
        )
).select(
    "category",
    "product",
    "total_revenue",
    "rank"
)


df_rank.show()



+-----------+------------+-------------+----+
|   category|     product|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      laptop|     247000.0|   1|
|Electronics|     monitor|      82650.0|   2|
|Electronics|  headphones|      19000.0|   3|
|Electronics|    keyboard|      10440.0|   4|
|  Furniture|        desk|      55200.0|   1|
|  Furniture|office chair|      39950.0|   2|
| Stationery|notebook set|       7900.0|   1|
+-----------+------------+-------------+----+



In [127]:
df_running_total = df_trim.withColumn(
    "running_total",
    sum("net_amount").over(
        Window.partitionBy("customer_id").orderBy("order_id","order_date")
    ))
df_running_total.show()
df_running_total.printSchema()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|running_total|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|      61750.0|
|    5004|       1001|2026-01-08|    keyboard|Electronics|       2|      1800|           5|           UPI|Completed|        3600|          180.0|    3420.0|             High|      65170.0|
|    5025|       1001|2026-03-02|      laptop|Electroni

In [128]:
df_running_total_1 = df_trim.withColumn(
    "previous_date",
    lag("order_date").over(
        Window.partitionBy("customer_id").orderBy(col("order_id").desc())
  )
).withColumn(
    "previous_amount",
    lag("net_amount").over(
        Window.partitionBy("customer_id").orderBy("order_date")
    )
).withColumn(
    "difference_amount",
    col("net_amount") - col("previous_amount")
)

df_running_total_1.show()


+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+---------------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|previous_date|previous_amount|difference_amount|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+---------------+-----------------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|   2026-01-08|           NULL|             NULL|
|    5004|       1001|2026-01-08|    keyboard|Electronics|       2|      1800|           5|           UPI|Co

In [129]:
df_latest_order = df_trim.withColumn(
    "latest_order_date",
    row_number().over(
        Window.partitionBy("customer_id").orderBy(col("order_date").desc())
    )
).filter(
    col("latest_order_date") == 1
)
df_latest_order.show()
df_latest_order.count()


+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|latest_order_date|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-----------------+
|    5025|       1001|2026-03-02|      laptop|Electronics|       1|     65000|           0|          Card|Completed|       65000|            0.0|   65000.0|              Low|                1|
|    5026|       1002|2026-03-04|office chair|  Furniture|       1|      8500|           5|           UPI|Completed|        8500|          425.0|    8075.0|             High|                1|
|    5027|       1003|2026-03-06|no

12

In [130]:
df_latest_order_1 = df_trim.groupBy("customer_id").agg(
    max(col("order_date")).alias("latest_order_date")
).orderBy(
    col("customer_id")
)

df_latest_order_1.show()
df_latest_order_1.count()

+-----------+-----------------+
|customer_id|latest_order_date|
+-----------+-----------------+
|       1001|       2026-03-02|
|       1002|       2026-03-04|
|       1003|       2026-03-06|
|       1004|       2026-02-05|
|       1005|       2026-03-08|
|       1006|       2026-02-09|
|       1007|       2026-03-10|
|       1008|       2026-02-14|
|       1009|       2026-02-18|
|       1010|       2026-03-12|
|       1011|       2026-02-22|
|       1012|       2026-02-25|
+-----------+-----------------+



12

**Section 6 — Spark SQL**

Q16. Register and query a view

Q17. SQL versus DataFrame API

In [137]:
df_trim.createOrReplaceTempView("sales_orders")

In [140]:
spark.sql(
    """
    select
      *,
      extract(month from order_date) as month,
      sum(net_amount) OVER (partition by category, extract(month from order_date)) as monthly_revenue
    from
      sales_orders
    where
      status = "Completed"

    """
).show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-----+---------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|month|monthly_revenue|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-----+---------------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|    1|       171045.0|
|    5002|       1002|2026-01-04|  headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|             High|    1|       171045.0|
|    5004|     

In [146]:
df_sql_test = df_trim.filter(
    col("status") == "Completed"
).withColumn(
    "month",
    expr("extract(month from order_date)")
).withColumn(
    "monthly_revenue",
    sum("net_amount").over(
        Window.partitionBy("category", "month")
    )
)

df_sql_test.show()


+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-----+---------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|order_value_brand|month|monthly_revenue|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-----+---------------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|              Low|    1|       171045.0|
|    5002|       1002|2026-01-04|  headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|             High|    1|       171045.0|
|    5004|     

**1. Readability**
* API: Uses Python functions and method chaining.
* SQL: Making queries easier to read for SQL developers.


**2. Maintainability**
* API: Better for integrating transformations with Python logic.
* SQL: Better for managing complex queries and reporting logic in SQL.


**3. Execution Planning**
* Conclusion: Both can deliver similar performance when they perform equivalent operations.



**Section 7 — Joins & Data Quality**

Q18. Join types and unmatched keys

Q19. Avoiding duplicate-column and row-count issues

Q20. End-to-end analysis challenge

In [159]:
df_joins_sql = df_trim.join(
    df_customer,
    on = "customer_id",
    how = "inner"
).orderBy(
    col("customer_id")
)

df_joins_sql.show(df_joins_sql.count(),truncate=False)


+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+----------+------+-------------+---------------+
|customer_id|order_id|order_date|product     |category   |quantity|unit_price|discount_pct|payment_method|status   |gross_amount|discount_amount|net_amount|order_value_brand|customer_name|city      |region|customer_tier|account_manager|
+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+----------+------+-------------+---------------+
|1001       |5001    |2026-01-03|laptop      |Electronics|1       |65000     |5           |Card          |Completed|65000       |3250.0         |61750.0   |Low              |Aarav        |Chennai   |South |Premium      |Ananya         |
|1001       |5004    |2026-01-08|keyboard    |Electr

In [160]:
df_joins_sql = df_trim.join(
    df_customer,
    on = "customer_id",
    how = "left"
).orderBy(
    col("customer_id")
)
df_joins_sql.show(df_joins_sql.count(),truncate=False)



+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+----------+------+-------------+---------------+
|customer_id|order_id|order_date|product     |category   |quantity|unit_price|discount_pct|payment_method|status   |gross_amount|discount_amount|net_amount|order_value_brand|customer_name|city      |region|customer_tier|account_manager|
+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+-------------+----------+------+-------------+---------------+
|1001       |5001    |2026-01-03|laptop      |Electronics|1       |65000     |5           |Card          |Completed|65000       |3250.0         |61750.0   |Low              |Aarav        |Chennai   |South |Premium      |Ananya         |
|1001       |5004    |2026-01-08|keyboard    |Electr

**1. Avoid duplicate columns:** Use aliases and select() to prevent ambiguous column references after joining.

**2. Duplicate keys will multiply rows:** Duplicate customer_id values in the customer table can match multiple times, increasing the joined row count.

**3. Check uniqueness:** Use groupBy("customer_id").count().filter(col("count") > 1) to identify duplicate customer IDs.

**4. Validate before joining:** Stop the join if duplicate customer IDs exist to prevent unintended row multiplication.

**5. Verify the result:** Compare row counts before and after joining and check for inflated revenue totals.